In [1]:
import pandas as pd
from pathlib import Path
import librosa
import os
import numpy as np

In [2]:
ROOT = Path.cwd().parent
DATA_DIR = ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
COUGHS_DIR = RAW_DIR / "coughs"
COUGHVID_DIR = COUGHS_DIR/ "COUGHVID_V3"
TB_DIR = COUGHS_DIR / "TBscreen_Dataset"
WEST_CHINA_DIR = COUGHS_DIR / "West_China_Uni"
BRONCHITIS_DIR = WEST_CHINA_DIR / "bronchitis"
PNEUMONIA_DIR = WEST_CHINA_DIR / "pneumonia"
ARTIFACTS_DIR = ROOT / "artifacts"

In [3]:
def load_csv(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    return pd.read_csv(path, low_memory=False)  # mixed-type columns, avoid dtype warnings


# COUGHVID_v3
coughvid_df = load_csv(COUGHVID_DIR / "tabular_form" / "coughvid_v3.csv")
coughvid_extr_features_df = load_csv(COUGHVID_DIR / "tabular_form" / "extracted_features_coughvid_v3.csv")
coughvid_filtered_exp_lbls = load_csv(COUGHVID_DIR / "tabular_form" / "filtered_expert_labels_coughvid_v3.csv")

# TB_screen
tb_forced_df = load_csv(TB_DIR / "Forced_coughs" / "Forced_coughs.csv")
tb_passive_df = load_csv(TB_DIR / "Passive_coughs" / "Passive_coughs.csv")
tb_metadata_df = load_csv(TB_DIR / "metadata.csv")

# West China, no metadata file, labels come from directory names
# (bronchitis / pneumonia), built below in section 3

In [4]:
coughvid_df.columns

Index(['datetime', 'cough_detected', 'latitude', 'longitude', 'age', 'gender',
       'respiratory_condition', 'fever_muscle_pain', 'status', 'file_name',
       'audio_name'],
      dtype='object')

In [5]:
coughvid_df.shape

(34434, 11)

In [6]:
coughvid_df['status'].value_counts()

status
healthy        15476
symptomatic     3873
COVID-19        1315
Name: count, dtype: int64

In [7]:
coughvid_df = coughvid_df.dropna(subset=["status"]).reset_index(drop=True)

In [8]:
coughvid_df.shape

(20664, 11)

In [9]:
coughvid_df['status'].value_counts()

status
healthy        15476
symptomatic     3873
COVID-19        1315
Name: count, dtype: int64

In [10]:
coughvid_df = coughvid_df[coughvid_df['cough_detected'] >= 0.4].reset_index(drop=True)

In [11]:
coughvid_df.shape

(16662, 11)

In [12]:
coughvid_df['status'].value_counts()

status
healthy        12498
symptomatic     3255
COVID-19         909
Name: count, dtype: int64

In [13]:
coughvid_df.columns

Index(['datetime', 'cough_detected', 'latitude', 'longitude', 'age', 'gender',
       'respiratory_condition', 'fever_muscle_pain', 'status', 'file_name',
       'audio_name'],
      dtype='object')

In [14]:
final_coughvid_df = coughvid_df.drop(columns=['datetime', 'age', 'gender', 'respiratory_condition', 'cough_detected', 'latitude', 'longitude', 'fever_muscle_pain', 'file_name'])

In [15]:
final_coughvid_df.columns

Index(['status', 'audio_name'], dtype='object')

In [16]:
covid = final_coughvid_df[final_coughvid_df['status'] == 'COVID-19']
healthy = final_coughvid_df[final_coughvid_df['status'] == 'healthy'].sample(n=1000, random_state=42)

final_coughvid_df = pd.concat([covid, healthy]).reset_index(drop=True)

In [17]:
final_coughvid_df.shape

(1909, 2)

In [18]:
final_coughvid_df.head()

,status,audio_name
0,COVID-19,72ab770e-a11d-4f98-8e94-7027f3f7a0ab.webm
1,COVID-19,6a2ffeef-99c5-4765-8dac-92aec8459d79.webm
2,COVID-19,4fa3ac8d-f252-40d6-bccd-24aa5e35556c.webm
3,COVID-19,21d95211-da92-44a9-83c7-30f8d4c6d670.webm
4,COVID-19,dbb52561-93ca-444e-9ff9-e61fd6c6996f.webm


In [19]:
def extract_features(file_path, sr=22050):
    try:
        y, sr = librosa.load(file_path, sr=sr)

        features = {}

        # MFCCs, capture timbre, very important for cough/voice classification
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
        for i in range(13):
            features[f'mfcc_{i}_mean'] = np.mean(mfccs[i])
            features[f'mfcc_{i}_std'] = np.std(mfccs[i])

        # Chroma, pitch class content
        chroma = librosa.feature.chroma_stft(y=y, sr=sr)
        features['chroma_mean'] = np.mean(chroma)
        features['chroma_std'] = np.std(chroma)

        # Spectral centroid, brightness of sound
        spec_centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
        features['spec_centroid_mean'] = np.mean(spec_centroid)
        features['spec_centroid_std'] = np.std(spec_centroid)

        # Spectral bandwidth
        spec_bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)
        features['spec_bandwidth_mean'] = np.mean(spec_bandwidth)
        features['spec_bandwidth_std'] = np.std(spec_bandwidth)

        # Spectral rolloff
        spec_rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)
        features['spec_rolloff_mean'] = np.mean(spec_rolloff)
        features['spec_rolloff_std'] = np.std(spec_rolloff)

        # Zero crossing rate, useful for distinguishing noisy vs tonal sounds
        zcr = librosa.feature.zero_crossing_rate(y)
        features['zcr_mean'] = np.mean(zcr)
        features['zcr_std'] = np.std(zcr)

        # RMS energy, loudness
        rms = librosa.feature.rms(y=y)
        features['rms_mean'] = np.mean(rms)
        features['rms_std'] = np.std(rms)

        # Spectral contrast
        contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
        features['contrast_mean'] = np.mean(contrast)
        features['contrast_std'] = np.std(contrast)

        # Tempo, sometimes useful, cough patterns have rhythm too
        tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
        features['tempo'] = tempo

        # Duration
        features['duration'] = librosa.get_duration(y=y, sr=sr)

        return features

    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

In [20]:
"""
Audio feature extraction pipeline for cough/respiratory classification.

Requires ffmpeg installed and on PATH for .webm/.ogg files:
    Windows: winget install ffmpeg  (then restart terminal/kernel)
    Mac:     brew install ffmpeg
    Linux:   sudo apt install ffmpeg
"""

import os
import warnings
import logging
import traceback
from pathlib import Path

import numpy as np
import pandas as pd
import librosa
from tqdm import tqdm

warnings.filterwarnings("ignore")

# Logging setup, writes failures to a file instead of flooding the console
logging.basicConfig(
    filename="feature_extraction_errors.log",
    level=logging.ERROR,
    format="%(asctime)s | %(message)s",
)

# ---------------------------------------------------------------------------
# Config, adjust these paths/columns to match your setup
# ---------------------------------------------------------------------------
COUGHVID_AUDIO_DIR = COUGHVID_DIR / "coughvid_20211012"
ID_COLUMN = "audio_name"        
FILE_EXTENSION = '.webm'         
CSV_CACHE_PATH = "final_df.csv"
SAMPLE_RATE = 22050


def build_file_path(audio_folder: Path, name: str, extension: str | None) -> Path:
    if extension and not str(name).endswith(extension):
        name = f"{name}{extension}"
    return audio_folder / name


def extract_features(clip_path: Path, sr: int = SAMPLE_RATE) -> dict | None:
    try:
        y, sr = librosa.load(clip_path, sr=sr)

        if y is None or len(y) == 0:
            logging.error(f"Empty audio after load: {clip_path}")
            return None

        features = {}

        # MFCCs, timbre, most important block for cough/voice classification
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
        for i, (mean, std) in enumerate(zip(mfccs.mean(axis=1), mfccs.std(axis=1))):
            features[f"mfcc_{i}_mean"] = mean
            features[f"mfcc_{i}_std"] = std

        # Chroma, pitch class content
        chroma = librosa.feature.chroma_stft(y=y, sr=sr)
        features["chroma_mean"] = chroma.mean()
        features["chroma_std"] = chroma.std()

        # Spectral centroid, brightness / center of mass of frequency
        centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
        features["spectral_centroid_mean"] = centroid.mean()
        features["spectral_centroid_std"] = centroid.std()

        # Spectral rolloff, frequency below which 85% of energy is contained
        rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)
        features["spectral_rolloff_mean"] = rolloff.mean()
        features["spectral_rolloff_std"] = rolloff.std()

        # Spectral bandwidth
        bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)
        features["bandwidth_mean"] = bandwidth.mean()
        features["bandwidth_std"] = bandwidth.std()

        # Spectral contrast
        contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
        features["contrast_mean"] = contrast.mean()
        features["contrast_std"] = contrast.std()

        # Zero crossing rate, helps distinguish dry vs wet cough
        zcr = librosa.feature.zero_crossing_rate(y)
        features["zcr_mean"] = zcr.mean()
        features["zcr_std"] = zcr.std()

        # RMS energy, loudness
        rms = librosa.feature.rms(y=y)
        features["rms_mean"] = rms.mean()
        features["rms_std"] = rms.std()

        # Duration
        features["duration"] = librosa.get_duration(y=y, sr=sr)

        return features

    except Exception:
        # Full traceback goes to the log file, keeps the console/tqdm bar clean
        logging.error(f"Failed on {clip_path}\n{traceback.format_exc()}")
        return None


def run_pipeline(df: pd.DataFrame) -> pd.DataFrame:
    if os.path.exists(CSV_CACHE_PATH):
        print(f"Loading cached features from {CSV_CACHE_PATH} ...")
        return pd.read_csv(CSV_CACHE_PATH)

    print("Extracting features, this will take a while on a large dataset...")

    audio_records = []
    missing_files = 0
    failed_files = 0

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Extracting features"):
        audio_path = build_file_path(COUGHVID_AUDIO_DIR, row[ID_COLUMN], FILE_EXTENSION)

        if not audio_path.exists():
            missing_files += 1
            logging.error(f"Missing file: {audio_path}")
            continue

        features = extract_features(audio_path)

        if features is None:
            failed_files += 1
            continue

        features[ID_COLUMN] = row[ID_COLUMN]
        audio_records.append(features)

    print(f"Done. {len(audio_records)} succeeded, {missing_files} missing, {failed_files} failed to decode.")
    print("See feature_extraction_errors.log for details on failures.")

    audio_df = pd.DataFrame(audio_records)
    final_df = df.merge(audio_df, on='audio_name', how="inner")

    final_df.to_csv(CSV_CACHE_PATH, index=False)
    print(f"Saved to {CSV_CACHE_PATH}")

    return final_df

In [21]:
print(COUGHVID_AUDIO_DIR)
print(final_coughvid_df['audio_name'].iloc[0])

test_path = build_file_path(COUGHVID_AUDIO_DIR, final_coughvid_df['audio_name'].iloc[0], FILE_EXTENSION)
print(test_path)
print(test_path.exists())

import os
print(os.listdir(COUGHVID_AUDIO_DIR)[:5])
final_df = run_pipeline(final_coughvid_df)
print(f"final_df shape: {final_coughvid_df.shape}")

c:\Users\emirl\respiratory-disease-screening\data\raw\coughs\COUGHVID_V3\coughvid_20211012
72ab770e-a11d-4f98-8e94-7027f3f7a0ab.webm
c:\Users\emirl\respiratory-disease-screening\data\raw\coughs\COUGHVID_V3\coughvid_20211012\72ab770e-a11d-4f98-8e94-7027f3f7a0ab.webm
True
['00014dcc-0f06-4c27-8c7b-737b18a2cf4c.json', '00014dcc-0f06-4c27-8c7b-737b18a2cf4c.webm', '00039425-7f3a-42aa-ac13-834aaa2b6b92.json', '00039425-7f3a-42aa-ac13-834aaa2b6b92.webm', '0007c6f1-5441-40e6-9aaf-a761d8f2da3b.json']
Extracting features, this will take a while on a large dataset...


Extracting features: 100%|██████████| 1909/1909 [03:40<00:00,  8.64it/s]


Done. 1677 succeeded, 232 missing, 0 failed to decode.
See feature_extraction_errors.log for details on failures.
Saved to final_df.csv
final_df shape: (1909, 2)


In [23]:
final_df.head()

,status,audio_name,mfcc_0_mean,mfcc_0_std,mfcc_1_mean,mfcc_1_std,mfcc_2_mean,mfcc_2_std,mfcc_3_mean,mfcc_3_std,...,spectral_rolloff_std,bandwidth_mean,bandwidth_std,contrast_mean,contrast_std,zcr_mean,zcr_std,rms_mean,rms_std,duration
0,COVID-19,72ab770e-a11d-4f98-8e94-7027f3f7a0ab.webm,-470.847931,190.612717,42.349316,62.147629,-20.277021,39.468700,15.817849,27.149414,...,2524.126719,1074.424121,1029.427529,16.049184,12.626969,0.087139,0.114379,0.030565,0.053363,9.72
1,COVID-19,6a2ffeef-99c5-4765-8dac-92aec8459d79.webm,-354.059265,170.827591,63.228638,67.722466,-43.616390,58.100597,2.691243,26.449554,...,1710.580791,1565.791825,507.983420,23.105411,14.382338,0.171478,0.095028,0.070727,0.105960,9.96
2,COVID-19,4fa3ac8d-f252-40d6-bccd-24aa5e35556c.webm,-488.905792,152.985077,8.413086,18.446627,-8.186396,19.812853,1.097989,10.272716,...,1544.407638,2953.664652,356.064460,19.292783,11.028386,0.346085,0.211492,0.015557,0.045500,9.60
3,COVID-19,21d95211-da92-44a9-83c7-30f8d4c6d670.webm,-532.129578,149.447968,44.484325,65.836685,-8.647761,29.667706,5.361758,18.435301,...,2472.470332,1658.632607,1010.359113,16.638784,11.042805,0.138599,0.153303,0.012546,0.029861,9.90
4,COVID-19,dbb52561-93ca-444e-9ff9-e61fd6c6996f.webm,-386.770660,206.173264,28.192722,38.605892,-1.467129,30.957123,22.833906,30.141464,...,729.172432,2125.658382,446.617215,23.104090,17.871928,0.405967,0.122298,0.065221,0.081394,9.84
